# Lab 4: an answer service that can prove where its answer came from

**Time: about 80 minutes. Work in pairs.** One person types, the other checks.

**What you will build.** A small service that takes a question, retrieves with the Lab 3 stack,
writes an answer, and returns that answer together with the exact pieces of text it used. You will
then break it on purpose, by making it cite a piece of text that was never retrieved, and watch the
check catch it. Last, you will try to make it refuse the five questions the documents cannot answer,
and find out that no single setting does it cleanly.

**How the cells work.** Each step has up to three cells: look at the real thing, make your call, then
see the measurement. Every `YOUR CALL` cell holds a working value, so the notebook runs top to bottom
before you edit anything.

**Three words used throughout.** *Grounding* means every sentence in the answer comes from retrieved
text rather than from the model's memory. A *citation* is the identifier of the exact piece of text
used, so a reader can go and check it. *Faithfulness* is the share of the answer that is supported by
the retrieved text, from 0 to 1.

In [ ]:
# Setup. This cell is the same in every lab notebook and has to run first.
import sys, pathlib, textwrap

# ROOT must point at the lab folder. You may have opened this notebook from the lab
# folder or from notebooks/, so check which, and step up one level if needed.
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# src/ holds the shared lab code. Putting it on the search path is what makes
# `from freshcart import ...` work in the cells below.
sys.path.insert(0, str(ROOT / "src"))

RAW = ROOT / "data" / "raw"     # the seven original documents. Nothing ever writes here.
print("lab folder:", ROOT)
print("corpus files:", len(list(RAW.glob("*"))))

## Step 1. Look at the whole response, not just the answer

**What you will do.** Run the next cell. It builds the service and asks one real question, then prints
the complete response as the service would return it over the network.

**What you should see.** Six fields: `answer`, `citations`, `confidence`, `retrieved`, `refused` and
`refusal_reason`. The answer quotes the current policy and ends with a short identifier in square
brackets. `citations` holds that same identifier. `retrieved` holds five identifiers, because
retrieval returned five pieces and the answer used one of them. `refused` is false.

**Why this step is here.** Most teams return a string and nothing else. Every hard question that comes
later, such as "where did that come from" and "why did it say that", is unanswerable without the other
five fields. The shape of this response is the deliverable of the lab, more than the answer text.

In [ ]:
# RagStack is the whole answer service in one object: it holds the Lab 3 retrieval
# stack, the grounding prompt, the answer writer and the citation check.
# faithfulness scores how much of an answer is supported by the retrieved text.
import json, statistics
from ragstack import RagStack
from rag import faithfulness
from labkit import load_golden, table, banner

svc = RagStack(False, threshold=0.15)    # threshold: refuse when the best score is below this
ans, hits = svc.ask("How many refund claims can a customer make in thirty days?")
print(json.dumps(ans.to_dict(), indent=1)[:900])
print("\nretrieved pieces, best first:")
for score, ch in hits:
    print(f"  {score:5.3f}  {ch.id}  {ch.meta['source']}")

## Step 2. Break it on purpose: a citation for text that was never retrieved

**What you will do.** Wrap the answer writer so that every third answer has an extra identifier added
to it, one that was never retrieved. Then run 25 questions and count how many the citation check
catches.

**What you should see.** 8 answers carry the fake identifier and all 8 are caught and refused, with
the reason starting `unverifiable citations`. The service refuses rather than returning a sentence
that points at nothing.

**Why this step is here.** A model that invents a source is the failure that ends up in the newspaper,
and the Air Canada case study on this day is exactly that. You cannot test for it by reading answers,
because invented citations look like real ones. You test for it by checking, in code, that every
identifier in the answer is one that retrieval actually returned.

In [ ]:
# The golden set is deliberately in two groups. Refusing an unanswerable question is
# right; refusing an answerable one is a failure. A single refusal rate hides both.
gold = load_golden("qa_golden.json")
questions = [(g, item) for g in ("answerable", "unanswerable") for item in gold[g]]
print(len(gold["answerable"]), "questions the documents can answer")
print(len(gold["unanswerable"]), "questions they cannot")

In [ ]:
# YOUR CALL: how many of the 25 answers do you think will carry a fake citation,
# and how many of those will be caught?
PREDICT_FAKES = 8
PREDICT_CAUGHT = 8

In [ ]:
# Fault injection: wrap the answer writer so every third answer carries an identifier
# that retrieval never returned. This is what an invented source looks like in code.
def phantom_every_third(generate):
    state = {"n": 0}
    def wrapped(prompt, context, question=None):
        text = generate(prompt, context, question)
        state["n"] += 1
        return text + " [deadbeef0000]" if state["n"] % 3 == 0 else text
    return wrapped

broken = RagStack(False, threshold=0.15)
broken.generate = phantom_every_third(broken.generate)
caught = 0
for group, item in questions:
    a, h = broken.ask(item["question"])
    if (a.refusal_reason or "").startswith("unverifiable citations"):
        caught += 1
print(f"answers carrying a fake citation, caught and refused: {caught}")
print(f"you predicted {PREDICT_CAUGHT} caught out of {PREDICT_FAKES} faked")
print("\nnote what did NOT happen: nothing raised an error, and no answer text looked wrong.")

## Step 3. Measure the honest service on all 25 questions

**What you will do.** Run the real service over the 20 answerable and 5 unanswerable questions at the
default setting, and print the four numbers that matter.

**What you should see.** Refusals 0 out of 20 on the answerable questions, which is what you want, and
0 out of 5 on the unanswerable ones, which is not. Faithfulness 1.000, because the offline answer
writer quotes retrieved text directly. Correct facts 17 out of 20.

**Why this step is here.** The service is grounded and still wrong about the thing that matters most:
it answers five questions the documents cannot answer. Grounding stops invention. It does not stop
answering when you should say "I do not know".

In [ ]:
# Ask all 25 questions and record four things per answer: was it refused, how good was
# the best retrieval score, how faithful is the text, and does it state the expected fact.
rows = []
for group, item in questions:
    a, h = svc.ask(item["question"])
    text = a.answer or ""
    rows.append({"id": item["id"], "group": group, "refused": a.refused,
                 "top score": round(h[0][0], 3) if h else 0.0,
                 "faithful": round(faithfulness(text, h), 3) if text else None,
                 "correct": (any(e.lower() in text.lower() for e in item.get("expect_any", []))
                             if group == "answerable" else None)})
print(table(rows, ["id", "group", "refused", "top score", "faithful", "correct"]))

ansb = [r for r in rows if r["group"] == "answerable"]
unans = [r for r in rows if r["group"] == "unanswerable"]
banner("the four numbers")
print(f"  refused, answerable   : {sum(r['refused'] for r in ansb)}/{len(ansb)}   (want LOW)")
print(f"  refused, unanswerable : {sum(r['refused'] for r in unans)}/{len(unans)}   (want HIGH)")
print(f"  faithfulness          : {statistics.mean(r['faithful'] for r in ansb if r['faithful']):.3f}")
print(f"  correct facts         : {sum(bool(r['correct']) for r in ansb)}/{len(ansb)}")

## Step 4. Try to fix refusal with a confidence threshold

**What you will do.** The service refuses when the best retrieval score is below a threshold. In the
`YOUR CALL` cell, choose the threshold you would ship. The reveal prints the two score ranges and then
sweeps the threshold so you can see what each choice costs.

**What you should see.** The 20 answerable questions score between 0.641 and 0.915. The 5 unanswerable
ones score 0.468, 0.562, 0.581, 0.620 and 0.740. The two ranges overlap, so the sweep shows 0.6
refusing 3 of 5 unanswerable and none of the answerable, while 0.7 catches 4 of 5 but starts refusing
3 real questions.

**Why this step is here.** There is no setting that catches all five without hurting real questions,
and that is the finding, not a failure of the lab. A retrieval score says how similar the text is, not
whether it answers the question. The fix is a rule the answer writer must follow, such as requiring
two independent sources, and that is what Lab 8 puts into code and gates on.

In [ ]:
# YOUR CALL: the refusal threshold you would ship
THRESHOLD = 0.6

In [ ]:
# Put the two groups of retrieval scores side by side. If they do not separate cleanly,
# no threshold can separate the questions either, whatever number you pick.
a_scores = sorted(r["top score"] for r in ansb)
u_scores = sorted(r["top score"] for r in unans)
print(f"answerable  : min {a_scores[0]:.3f}  median {a_scores[len(a_scores)//2]:.3f}  max {a_scores[-1]:.3f}")
print(f"unanswerable: {u_scores}")
print("\nthe ranges overlap, so no single cut separates them:\n")
sweep = []
for t in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    sweep.append({"threshold": t,
                  "refused answerable": f"{sum(1 for s in a_scores if s < t)}/20",
                  "refused unanswerable": f"{sum(1 for s in u_scores if s < t)}/5"})
print(table(sweep, ["threshold", "refused answerable", "refused unanswerable"]))
print(f"\nyour choice was {THRESHOLD}: "
      f"{sum(1 for s in a_scores if s < THRESHOLD)}/20 answerable refused, "
      f"{sum(1 for s in u_scores if s < THRESHOLD)}/5 unanswerable refused")

## Step 5. Faithful, and still no use

**What you will do.** Print the answers that are fully grounded and still fail to state the fact the
question asked for.

**What you should see.** Three of them, including A14, "What is the first response TAT for a P2 case",
where the answer returns the title of the SLA document and an identifier, but never the number of
minutes.

**Why this step is here.** Faithfulness of 1.000 looks like a perfect score and is not one. It only
says the answer came from the documents. Whether the answer is any use to the agent reading it is a
different measurement, which is why the golden set stores the facts a correct answer must contain.
Take one of these three to the debrief and say whether the fault is retrieval, chunking or the answer
writer.

In [ ]:
# Grounded but useless: high faithfulness, and still missing the fact the question asked
# for. Faithfulness measures where the words came from, not whether they answer anything.
unhelpful = [r for r in ansb if (r["faithful"] or 0) >= 0.8 and not r["correct"]]
for r in unhelpful:
    item = next(i for i in gold["answerable"] if i["id"] == r["id"])
    a, h = svc.ask(item["question"])
    print(f"{r['id']}  Q: {item['question']}")
    print(f"      A: {(a.answer or '')[:110]}")
    print(f"      expected one of: {item.get('expect_any')}\n")
print(len(unhelpful), "answers are grounded but do not answer the question")

## When you are done

- Fill in the **Your write-up** cell at the bottom of this notebook: the four numbers, your threshold and what it costs, and which
  of the three unhelpful answers you would fix first.
- Run it as a real service: `python labs\lab04_rag_api.py serve`, then open
  `http://127.0.0.1:8000/docs` and ask it a question from the browser.
- Run it as a service on your own computer: see **Lab 4 as an API** in the README.

## Your write-up

**Type your answers into this cell.** Double click it to edit, then click away to render it again.
Everything you write here leaves with the notebook when you export it, and that export is what you
hand in.

Do not write in `handouts/lab4_answers.md`. It is the blank answer sheet, and a fresh copy of the repository replaces it.

---

**Pair:**

### 1. The response contract

List the six fields your service returns, and for each one, who needs it and why.

| Field | Who needs it | Why |
|---|---|---|
| | | |

### 2. The four numbers, reported separately

| Measure | Value | Should be |
|---|---|---|
| Refusal rate, answerable questions | /20 | low |
| Refusal rate, unanswerable questions | /5 | high |
| Faithfulness of answered questions | | near 1 |
| Correct facts, answerable questions | /20 | high |

### 3. Phantom citations

How many fake citations were caught? ___
How would you have discovered this in production, if the check did not exist?

### 4. The threshold you would ship

Threshold: ___  At that setting it refuses ___/20 answerable and ___/5 unanswerable.

Why is there no setting that catches all five without refusing real questions?

### 5. Grounded but useless

Pick one answer with faithfulness 1.000 that still fails the question.

Question id: ___  Where does the fault sit: retrieval, chunking, or the answer writer?

### 6. What you would fix first, and how you would prove it worked